In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   TS-FORECASTING KAGGLE — v38  (DiffRoC features + Diagnostic Plots)      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  BASE = v36 (score 0.2554 trên Kaggle):                                    ║
║    ✓ VAL_THRESHOLD = 3500  (GIỮ NGUYÊN — đây là lý do score cao)          ║
║    ✓ Per-horizon models (h=1,3,10,25 riêng biệt)                           ║
║    ✓ 2-stage: mdl_val → best_iter → mdl_full (không kfold)                ║
║    ✗ REMOVE: Neutralization (v37 overfit nặng, Kaggle 0.16)                ║
║    ✗ REMOVE: Time Decay (v37 thay đổi dynamics → generalization kém)      ║
║                                                                              ║
║  KỸ THUẬT MỚI DUY NHẤT: Differencing & Rate of Change (RoC)               ║
║  ─────────────────────────────────────────────────────────────────────      ║
║  Lý do chọn DiffRoC (tiềm năng nhất trong 4 kỹ thuật):                    ║
║    1. v36 chỉ có diff(1) — thiếu velocity ở nhiều timeframe               ║
║    2. RoC normalized giúp model so sánh magnitude thay đổi cross-series    ║
║    3. Acceleration (2nd derivative) capture regime change signal           ║
║    4. Không tăng nhiều features → RAM an toàn (~22GB)                     ║
║                                                                              ║
║  Features thêm cho mỗi source col:                                         ║
║    - diff(3), diff(5)   : sai phân ở 3 và 5 bước                         ║
║    - roc(5), roc(10)    : (x_t - x_{t-k}) / (|x_{t-k}| + eps)            ║
║    - accel              : x_t - 2*x_{t-1} + x_{t-2}  (curvature)         ║
║  5 source cols × 5 features mới = +25 cols (tổng ~193)                    ║
║                                                                              ║
║  OUTPUT: submission.csv + 4 diagnostic PNGs                                ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import gc
import os
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import lightgbm as lgb
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — tránh memory leak trên Kaggle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
T0 = time.time()

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG (GIỮ NGUYÊN SO VỚI v36)
# ══════════════════════════════════════════════════════════════════════════════
TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH  = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH, TEST_PATH = 'train.parquet', 'test.parquet'

# ── Threshold này là lý do score cao: val ≈ test distribution ──────────────
# KHÔNG THAY ĐỔI VAL_THRESHOLD
VAL_THRESHOLD  = 3500
HORIZONS       = [1, 3, 10, 25]
SEEDS          = [42, 2024, 12345]
CLIP_Q_LOW     = 0.005
CLIP_Q_HIGH    = 0.995

# LightGBM params (giữ nguyên v36 — đã được validate)
LGB_PARAMS = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.015,
    'n_estimators':      5000,
    'num_leaves':        90,
    'min_child_samples': 200,
    'feature_fraction':  0.65,
    'bagging_fraction':  0.75,
    'bagging_freq':      5,
    'lambda_l1':         0.1,
    'lambda_l2':        10.0,
    'verbosity':         -1,
    'n_jobs':            -1,
}

# ══════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════════════════════
def _clip01(x): return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    """Metric chính thức của competition — giữ NGUYÊN từ v36."""
    y_target = np.asarray(y_target, dtype=np.float64)
    y_pred   = np.asarray(y_pred,   dtype=np.float64)
    w        = np.asarray(w,        dtype=np.float64)
    denom    = np.sum(w * y_target**2)
    if denom <= 0:
        return 0.0
    ratio = np.sum(w * (y_target - y_pred)**2) / denom
    return float(np.sqrt(1.0 - _clip01(ratio)))


def log_ram(label=''):
    try:
        import psutil
        mb = psutil.Process(os.getpid()).memory_info().rss / 1024**2
        print(f'   [RAM {label}]: {mb:,.0f} MB ({mb/1024:.1f} GB)')
    except ImportError:
        pass


# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════
def build_features_polars(train_path, test_path):
    print('⏳ Feature Engineering v38 (DiffRoC)...')

    # ── Đọc & concat ─────────────────────────────────────────────────────────
    df_train = pl.read_parquet(train_path)
    df_test  = pl.read_parquet(test_path)

    df_train = df_train.with_columns(pl.lit(1, dtype=pl.Int8).alias('is_train'))
    df_test  = df_test.with_columns(
        pl.lit(0, dtype=pl.Int8).alias('is_train'),
        pl.lit(None).cast(pl.Float64).alias('y_target'),
        pl.lit(None).cast(pl.Float64).alias('weight'),
    ).select(df_train.columns)

    df = pl.concat([df_train, df_test]).sort(
        ['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    del df_train, df_test
    gc.collect()
    print(f'   concat done: {len(df):,} rows | {df.width} cols')
    log_ram('after concat')

    # ── Target Encoding (fit chỉ trên train ≤ 3500 — KHÔNG LEAK) ─────────────
    # NGUYÊN TẮC: bất kỳ aggregate nào cũng chỉ được tính trên data lịch sử
    # ts_index ≤ 3500 = "quá khứ", ts_index > 3500 = "tương lai/test"
    fit_df      = df.filter((pl.col('is_train') == 1) &
                            (pl.col('ts_index') <= VAL_THRESHOLD))
    global_mean = float(fit_df.select(pl.col('y_target').mean()).item())

    cat_enc  = fit_df.group_by('sub_category').agg(
        pl.col('y_target').mean().alias('sub_category_enc'))
    code_enc = fit_df.group_by('sub_code').agg(
        pl.col('y_target').mean().alias('sub_code_enc'))

    df = (df.join(cat_enc, on='sub_category', how='left')
            .with_columns(pl.col('sub_category_enc').fill_null(global_mean)))
    df = (df.join(code_enc, on='sub_code', how='left')
            .with_columns(pl.col('sub_code_enc').fill_null(global_mean)))
    del fit_df, cat_enc, code_enc
    gc.collect()

    # ── Interactions & CS z-score (giữ nguyên v36) ───────────────────────────
    exprs = []
    if 'feature_al' in df.columns and 'feature_am' in df.columns:
        exprs += [
            (pl.col('feature_al') - pl.col('feature_am')).alias('d_al_am'),
            (pl.col('feature_al') / (pl.col('feature_am') + 1e-7)).alias('r_al_am'),
        ]
    if 'feature_cg' in df.columns and 'feature_by' in df.columns:
        exprs.append((pl.col('feature_cg') - pl.col('feature_by')).alias('d_cg_by'))
    if exprs:
        df = df.with_columns(exprs)
        exprs.clear()

    cs_cols = ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am']
    for c in cs_cols:
        if c in df.columns:
            exprs.append(
                ((pl.col(c) - pl.col(c).mean().over('ts_index')) /
                 (pl.col(c).std().over('ts_index') + 1e-7)).alias(f'{c}_cs'))

    # ── Time cyclical (giữ nguyên v36) ───────────────────────────────────────
    exprs += [
        (np.sin(2 * np.pi * pl.col('ts_index') / 100.0)).alias('t_sin'),
        (np.cos(2 * np.pi * pl.col('ts_index') / 100.0)).alias('t_cos'),
    ]
    df = df.with_columns(exprs)
    exprs.clear()

    # ── Lag + Rolling + [NEW] DiffRoC ─────────────────────────────────────────
    target_cols = [c for c in
        ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'feature_s']
        if c in df.columns]
    group_cols = ['code', 'sub_code', 'sub_category', 'horizon']

    for c in target_cols:
        # ── v36 features (giữ nguyên) ─────────────────────────────────────
        for lag in [1, 3, 5, 10, 25]:
            exprs.append(pl.col(c).shift(lag).over(group_cols).alias(f'{c}_lag{lag}'))
        for w in [5, 10, 20]:
            exprs.append(pl.col(c).rolling_mean(w, min_periods=1).over(group_cols)
                           .alias(f'{c}_roll_mean_{w}'))
            exprs.append(pl.col(c).rolling_std(w, min_periods=1).over(group_cols)
                           .alias(f'{c}_roll_std_{w}'))
        exprs.append(pl.col(c).ewm_mean(span=10, min_periods=1, ignore_nulls=True)
                       .over(group_cols).alias(f'{c}_ewm_10'))
        exprs.append(pl.col(c).diff(1).over(group_cols).alias(f'{c}_diff1'))
        exprs.append((pl.col(c).rank() / pl.count()).over('ts_index')
                       .alias(f'{c}_rank'))

        # ── [NEW] DiffRoC features ────────────────────────────────────────
        # diff(3), diff(5): velocity ở timeframe dài hơn
        exprs.append(pl.col(c).diff(3).over(group_cols)
                       .cast(pl.Float32).alias(f'{c}_diff3'))
        exprs.append(pl.col(c).diff(5).over(group_cols)
                       .cast(pl.Float32).alias(f'{c}_diff5'))

        # roc(5), roc(10): normalized rate of change
        # Công thức: (x_t - x_{t-k}) / (|x_{t-k}| + eps)
        # Normalized → có thể so sánh magnitude giữa các series khác scale
        for k in [5, 10]:
            shifted = pl.col(c).shift(k).over(group_cols)
            roc_expr = ((pl.col(c) - shifted) / (shifted.abs() + 1e-7))
            # Clip ở ±5 để tránh inf khi denominator gần 0
            exprs.append(roc_expr.clip(-5.0, 5.0)
                           .cast(pl.Float32).alias(f'{c}_roc{k}'))

        # acceleration: x_t - 2*x_{t-1} + x_{t-2}  (second derivative)
        # Capture momentum thay đổi: đang tăng tốc hay giảm tốc?
        accel_expr = (
            pl.col(c)
            - 2 * pl.col(c).shift(1).over(group_cols)
            + pl.col(c).shift(2).over(group_cols)
        ).cast(pl.Float32).alias(f'{c}_accel')
        exprs.append(accel_expr)

    df = df.with_columns(exprs)
    exprs.clear()
    print(f'   Lag+DiffRoC done: {len(target_cols)} source cols')

    # ── Ép Float32, fill NaN → 0 ─────────────────────────────────────────────
    df = df.with_columns(cs.float().cast(pl.Float32).fill_null(0.0))

    print(f'   Total: {df.width} cols')
    log_ram('before to_pandas')

    # ── Tách train/test → Pandas, xóa Polars ngay ────────────────────────────
    df_train_pd = df.filter(pl.col('is_train') == 1).drop('is_train').to_pandas()
    df_test_pd  = (df.filter(pl.col('is_train') == 0)
                     .drop(['is_train', 'y_target', 'weight'])
                     .to_pandas())
    del df
    gc.collect()
    log_ram('FE done')

    return df_train_pd, df_test_pd


# ══════════════════════════════════════════════════════════════════════════════
# TRAIN PER-HORIZON (2-stage: val → full, giữ nguyên v36)
# ══════════════════════════════════════════════════════════════════════════════
def solve_horizon(horizon, train_df, test_df):
    """
    2-stage training:
      Stage 1: train trên ts≤3500, val trên ts>3500 → best_iteration
      Stage 2: train trên TOÀN BỘ với best_iteration đó → predict test

    Returns dict kèm feature_importance và OOF data cho diagnostic plots.
    """
    t_h = time.time()
    print(f"\n{'='*70}\n 🚀 HORIZON {horizon}\n{'='*70}")

    tr = train_df[train_df['horizon'] == horizon].copy()
    te = test_df[test_df['horizon'] == horizon].copy()

    EXCL  = {'id', 'code', 'sub_code', 'sub_category',
             'horizon', 'ts_index', 'weight', 'y_target'}
    feats = [c for c in tr.columns if c not in EXCL]

    # In stats về DiffRoC features
    n_diffroc = sum(1 for c in feats
                    if any(x in c for x in ['_diff3','_diff5','_roc','_accel']))
    print(f'Data: train={len(tr):,}, test={len(te):,} | '
          f'Features: {len(feats)} (DiffRoC={n_diffroc})')

    # ── Split ─────────────────────────────────────────────────────────────────
    fit_m = tr['ts_index'] <= VAL_THRESHOLD
    val_m = tr['ts_index'] >  VAL_THRESHOLD

    X_fit = tr.loc[fit_m, feats];   y_fit = tr.loc[fit_m, 'y_target']
    w_fit = tr.loc[fit_m, 'weight']
    X_val = tr.loc[val_m, feats];   y_val = tr.loc[val_m, 'y_target']
    w_val = tr.loc[val_m, 'weight']
    X_all = tr[feats];              y_all = tr['y_target']
    w_all = tr['weight']
    ts_val = tr.loc[val_m, 'ts_index'].values  # cho temporal residual plot

    val_pred   = np.zeros(len(y_val), dtype=np.float64)
    test_pred  = np.zeros(len(te),    dtype=np.float64)
    best_iters = []
    fi_cum     = np.zeros(len(feats), dtype=np.float64)  # feature importance

    for i, seed in enumerate(SEEDS, 1):
        print(f'   -> Seed {i}/{len(SEEDS)} (seed={seed})...')

        # Stage 1: val model → best_iteration
        mdl_val = lgb.LGBMRegressor(**LGB_PARAMS, random_state=seed)
        mdl_val.fit(
            X_fit, y_fit, sample_weight=w_fit,
            eval_set=[(X_val, y_val)], eval_sample_weight=[w_val],
            callbacks=[
                lgb.early_stopping(200, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )
        bi = max(int(mdl_val.best_iteration_ or LGB_PARAMS['n_estimators']), 20)
        best_iters.append(bi)
        val_pred += mdl_val.predict(X_val) / len(SEEDS)
        del mdl_val
        gc.collect()

        # Stage 2: full model → test predictions
        mdl_full = lgb.LGBMRegressor(
            **{**LGB_PARAMS, 'n_estimators': bi}, random_state=seed)
        mdl_full.fit(
            X_all, y_all, sample_weight=w_all,
            callbacks=[lgb.log_evaluation(period=-1)],
        )
        test_pred += mdl_full.predict(te[feats]) / len(SEEDS)
        fi_cum    += mdl_full.feature_importances_
        del mdl_full
        gc.collect()

    # ── Score & clip ──────────────────────────────────────────────────────────
    score      = weighted_rmse_score(y_val.values, val_pred, w_val.values)
    q_lo, q_hi = np.quantile(y_fit.values, [CLIP_Q_LOW, CLIP_Q_HIGH])
    test_clip  = np.clip(test_pred, q_lo, q_hi)

    print(f'💡 h={horizon} | score={score:.6f} | '
          f'avg_iter={np.mean(best_iters):.0f} | '
          f'elapsed={(time.time()-t_h)/60:.1f} min')

    # ── Build feature importance DataFrame ───────────────────────────────────
    fi_df = pd.DataFrame({'feature': feats, 'importance': fi_cum / len(SEEDS)})\
              .sort_values('importance', ascending=False).reset_index(drop=True)

    out = {
        'horizon':        horizon,
        'id_test':        te['id'].values,
        'test_pred_raw':  test_pred,
        'test_pred_clip': test_clip,
        'id_val':         tr.loc[val_m, 'id'].values,
        'y_val':          y_val.values,
        'w_val':          w_val.values,
        'val_pred':       val_pred,
        'ts_val':         ts_val,
        'score_local':    score,
        'feature_imp':    fi_df,          # dùng cho diagnostic plots
    }

    # Xóa ngay sau khi xong horizon này
    del tr, te, X_fit, y_fit, w_fit, X_val, y_val, w_val, X_all, y_all, w_all
    del val_pred, test_pred, test_clip, fi_cum, fi_df, ts_val
    gc.collect()

    return out


# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC PLOTS
# ══════════════════════════════════════════════════════════════════════════════
def create_diagnostic_plots(all_outputs, oof):
    """
    Tạo 4 bộ biểu đồ để phát hiện vấn đề:
      Plot 1: Score per horizon + comparison v36
      Plot 2: OOF Pred vs True scatter (4 horizons)
      Plot 3: Residual mean theo thời gian (temporal bias)
      Plot 4: Feature importance top 15 (4 horizons)

    Mỗi plot save riêng → dễ đọc trên Kaggle output viewer.
    """
    print('\n📊 Generating diagnostic plots...')

    HC = {1: '#2196F3', 3: '#4CAF50', 10: '#FF9800', 25: '#E91E63'}
    V36_SCORES = {1: 0.0817, 3: 0.1427, 10: 0.2117, 25: 0.2706}  # reference
    DARK = '#111827'

    # ── Plot 1: Score comparison ──────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor(DARK)
    ax.set_facecolor('#1f2937')

    h_list = sorted([o['horizon'] for o in all_outputs])
    s_v38  = [next(o['score_local'] for o in all_outputs if o['horizon'] == h)
              for h in h_list]
    s_v36  = [V36_SCORES[h] for h in h_list]
    x      = np.arange(len(h_list))

    ax.bar(x - 0.2, s_v36, 0.35, label='v36 (0.2554 Kaggle)', color='#4B5563',
           edgecolor='#6B7280')
    ax.bar(x + 0.2, s_v38, 0.35, label='v38',
           color=[HC[h] for h in h_list], edgecolor='#9CA3AF')
    for i, (sv36, sv38, h) in enumerate(zip(s_v36, s_v38, h_list)):
        diff = sv38 - sv36
        sign = '+' if diff >= 0 else ''
        ax.text(i + 0.2, sv38 + 0.003, f'{sign}{diff:.4f}',
                ha='center', va='bottom', fontsize=9,
                color='#4ade80' if diff >= 0 else '#f87171')

    ax.set_xticks(x)
    ax.set_xticklabels([f'h={h}' for h in h_list], color='#D1D5DB')
    ax.tick_params(colors='#9CA3AF')
    ax.set_ylabel('Local Score', color='#D1D5DB')
    ax.set_title('Score per Horizon: v36 vs v38', color='#F9FAFB', pad=10)
    ax.legend(fontsize=9, facecolor='#374151', labelcolor='#D1D5DB',
               edgecolor='#4B5563')
    for sp in ax.spines.values():
        sp.set_edgecolor('#374151')
    plt.tight_layout()
    plt.savefig('diag1_scores.png', dpi=130, bbox_inches='tight',
                facecolor=DARK)
    plt.close()
    print('   ✓ diag1_scores.png')

    # ── Plot 2: OOF Pred vs True scatter ──────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    fig.patch.set_facecolor(DARK)

    for ax, out in zip(axes, sorted(all_outputs, key=lambda o: o['horizon'])):
        h = out['horizon']
        ax.set_facecolor('#1f2937')
        yt = out['y_val']
        yp = out['val_pred']
        # Sample tối đa 3000 điểm để vẽ nhanh
        n  = min(3000, len(yt))
        idx = np.random.choice(len(yt), n, replace=False)
        ax.scatter(yt[idx], yp[idx], s=4, alpha=0.3, color=HC[h])
        lo = min(yt.min(), yp.min())
        hi = max(yt.max(), yp.max())
        ax.plot([lo, hi], [lo, hi], 'w--', lw=1, alpha=0.5)
        ax.set_title(f'h={h}  score={out["score_local"]:.4f}',
                     color='#F9FAFB', fontsize=10)
        ax.set_xlabel('y_true', color='#9CA3AF', fontsize=8)
        ax.set_ylabel('y_pred', color='#9CA3AF', fontsize=8)
        ax.tick_params(colors='#6B7280', labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')

    plt.suptitle('OOF Pred vs True — Đường chéo = perfect prediction',
                 color='#D1D5DB', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('diag2_pred_vs_true.png', dpi=130, bbox_inches='tight',
                facecolor=DARK)
    plt.close()
    print('   ✓ diag2_pred_vs_true.png')

    # ── Plot 3: Residual theo thời gian (temporal bias check) ─────────────────
    # Đây là biểu đồ quan trọng nhất để phát hiện regime shift
    # Nếu residual phẳng → model ổn
    # Nếu có spike/trend → model bị regime shift tại điểm đó
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.patch.set_facecolor(DARK)

    for ax, out in zip(axes, sorted(all_outputs, key=lambda o: o['horizon'])):
        h   = out['horizon']
        ax.set_facecolor('#1f2937')
        ts  = out['ts_val']
        res = out['y_val'] - out['val_pred']

        # Bin theo 50 khoảng ts_index
        bins = np.percentile(ts, np.linspace(0, 100, 51))
        bx, by, be = [], [], []
        for j in range(len(bins)-1):
            m = (ts >= bins[j]) & (ts < bins[j+1])
            if m.sum() > 20:
                bx.append((bins[j] + bins[j+1]) / 2)
                by.append(np.mean(res[m]))
                be.append(np.std(res[m]) / np.sqrt(m.sum()))

        bx, by, be = map(np.array, [bx, by, be])
        ax.plot(bx, by, color=HC[h], lw=1.5)
        ax.fill_between(bx, by - be, by + be, alpha=0.2, color=HC[h])
        ax.axhline(0, color='white', ls='--', lw=0.8, alpha=0.5)
        ax.set_title(f'h={h}  temporal residual', color='#F9FAFB', fontsize=10)
        ax.set_xlabel('ts_index', color='#9CA3AF', fontsize=8)
        ax.set_ylabel('mean residual', color='#9CA3AF', fontsize=8)
        ax.tick_params(colors='#6B7280', labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')

    plt.suptitle('Residual theo thời gian — Phẳng = tốt | Spike = regime shift',
                 color='#D1D5DB', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('diag3_temporal_residual.png', dpi=130, bbox_inches='tight',
                facecolor=DARK)
    plt.close()
    print('   ✓ diag3_temporal_residual.png')

    # ── Plot 4: Feature importance top 15 ─────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor(DARK)
    axes_flat = axes.flatten()

    for ax, out in zip(axes_flat, sorted(all_outputs, key=lambda o: o['horizon'])):
        h     = out['horizon']
        fi    = out['feature_imp'].head(15)
        ax.set_facecolor('#1f2937')

        # Tô màu khác nhau cho DiffRoC features (mới) vs v36 features (cũ)
        colors = []
        for feat in fi['feature']:
            if any(x in feat for x in ['_diff3', '_diff5', '_roc', '_accel']):
                colors.append('#f59e0b')  # amber = DiffRoC mới
            else:
                colors.append(HC[h])      # màu horizon = v36 feature

        ax.barh(range(len(fi)), fi['importance'], color=colors, edgecolor='#374151')
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi['feature'], fontsize=7.5, color='#D1D5DB')
        ax.invert_yaxis()
        ax.set_title(f'h={h}  Feature Importance Top 15',
                     color='#F9FAFB', fontsize=11)
        ax.set_xlabel('importance', color='#9CA3AF', fontsize=8)
        ax.tick_params(colors='#6B7280', labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')

        # Legend: màu vàng = DiffRoC mới, màu horizon = v36
        from matplotlib.patches import Patch
        legend_elems = [
            Patch(facecolor='#f59e0b', label='DiffRoC (new)'),
            Patch(facecolor=HC[h],     label='v36 feature'),
        ]
        ax.legend(handles=legend_elems, fontsize=8,
                  facecolor='#374151', labelcolor='#D1D5DB',
                  edgecolor='#4B5563', loc='lower right')

    plt.suptitle(
        'Feature Importance — Vàng = DiffRoC mới, Màu = v36 feature\n'
        'Nếu vàng nhiều ở top → DiffRoC thực sự hữu ích',
        color='#D1D5DB', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('diag4_feature_importance.png', dpi=130, bbox_inches='tight',
                facecolor=DARK)
    plt.close()
    print('   ✓ diag4_feature_importance.png')

    print('\n📊 Cách đọc plots:')
    print('  diag1: Cột xanh lá = cải thiện so v36, đỏ = giảm')
    print('  diag2: Điểm càng gần đường chéo trắng → model càng chính xác')
    print('  diag3: Spike tại ts nào → regime shift tại đó → cần model riêng cho vùng đó')
    print('  diag4: Features vàng (DiffRoC) top → kỹ thuật có tác dụng')


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':

    # Phase 1: Feature Engineering
    df_train_all, df_test_all = build_features_polars(TRAIN_PATH, TEST_PATH)
    log_ram('after FE')

    # Phase 2: Train per horizon
    all_outputs = []
    for h in HORIZONS:
        all_outputs.append(solve_horizon(h, df_train_all, df_test_all))

    # Phase 3: Tổng hợp submission
    sub_raw_parts, sub_clip_parts, oof_parts = [], [], []
    for out in all_outputs:
        sub_raw_parts.append(
            pd.DataFrame({'id': out['id_test'], 'prediction': out['test_pred_raw']}))
        sub_clip_parts.append(
            pd.DataFrame({'id': out['id_test'], 'prediction': out['test_pred_clip']}))
        oof_parts.append(pd.DataFrame({
            'id': out['id_val'], 'y_true': out['y_val'],
            'y_pred': out['val_pred'], 'w': out['w_val'],
            'horizon': out['horizon'],
        }))

    sub_raw  = pd.concat(sub_raw_parts,  ignore_index=True)
    sub_clip = pd.concat(sub_clip_parts, ignore_index=True)
    oof      = pd.concat(oof_parts,      ignore_index=True)
    del sub_raw_parts, sub_clip_parts, oof_parts
    gc.collect()

    # Đảm bảo thứ tự test gốc
    test_order = pd.read_parquet(TEST_PATH, columns=['id'])
    sub_clip   = test_order.merge(sub_clip, on='id', how='left').fillna(0)
    sub_raw    = test_order.merge(sub_raw,  on='id', how='left').fillna(0)
    del test_order

    agg_local = weighted_rmse_score(
        oof['y_true'].values, oof['y_pred'].values, oof['w'].values)

    # Phase 4: Diagnostic plots
    create_diagnostic_plots(all_outputs, oof)

    # Lưu files
    sub_clip.to_csv('submission.csv',        index=False)
    sub_raw.to_csv( 'submission_noclip.csv', index=False)
    oof.to_csv(     'oof_v38.csv',           index=False)

    # ── Final report ─────────────────────────────────────────────────────────
    V36_REF = {1: 0.0817, 3: 0.1427, 10: 0.2117, 25: 0.2706}
    print(f"\n{'='*70}")
    print(f'🏆 TỔNG KẾT v38 (DiffRoC) — ref v36 Kaggle=0.2554')
    print(f"{'─'*70}")
    print(f'  {"horizon":>8} | {"v36":>8} | {"v38":>8} | {"delta":>8}')
    print(f'  {"─"*8}-+-{"─"*8}-+-{"─"*8}-+-{"─"*8}')
    for out in sorted(all_outputs, key=lambda o: o['horizon']):
        h   = out['horizon']
        s38 = out['score_local']
        s36 = V36_REF[h]
        d   = s38 - s36
        tag = '↑' if d > 0 else '↓'
        print(f'  h={h:>6} | {s36:>8.4f} | {s38:>8.4f} | {tag}{abs(d):.4f}')
    print(f"{'─'*70}")
    print(f'🔥 Aggregate Local Score : {agg_local:.6f}')
    print(f'   [ref v36 local = 0.2317 | kaggle = 0.2554]')
    print(f"⏱️  Tổng thời gian : {(time.time()-T0)/60:.1f} phút")
    print(f"{'='*70}")

    print("""
CHECKLIST v38:

  [DiffRoC CÓ HIỆU QUẢ?]
  → diag4: features amber (_roc, _accel, _diff3/5) nằm trong top 10?
       Có → DiffRoC bổ sung real signal → giữ, thêm vào v39
       Không → DiffRoC redundant với lag/diff1 có sẵn → bỏ

  [REGIME SHIFT]
  → diag3: Spike lớn tại ts nào?
       Nếu spike ở ts_index ~ 3500 → đây là boundary effect bình thường
       Nếu spike ở nhiều điểm → bài toán có nhiều regimes

  [H=1 VẪN THẤP?]
  → y_target h=1 ≈ 0 → score metric degenerate, không phải lỗi model
  → Không cần fix, tập trung vào h=3,10,25

  [NẾU LOCAL TĂNG NHƯNG KAGGLE KHÔNG TĂNG]
  → DiffRoC có thể overfit val set
  → Bước tiếp theo: thử EMA multi-span (tiềm năng thứ 2)
""")

⏳ Feature Engineering v38 (DiffRoC)...
   concat done: 6,784,521 rows | 95 cols
   [RAM after concat]: 16,042 MB (15.7 GB)
   Lag+DiffRoC done: 5 source cols
   Total: 202 cols
   [RAM before to_pandas]: 22,716 MB (22.2 GB)
   [RAM FE done]: 28,672 MB (28.0 GB)
   [RAM after FE]: 28,672 MB (28.0 GB)

 🚀 HORIZON 1
Data: train=1,394,653, test=379,617 | Features: 193 (DiffRoC=25)
   -> Seed 1/3 (seed=42)...
   -> Seed 2/3 (seed=2024)...
   -> Seed 3/3 (seed=12345)...
💡 h=1 | score=0.072509 | avg_iter=259 | elapsed=12.0 min

 🚀 HORIZON 3
Data: train=1,385,816, test=376,558 | Features: 193 (DiffRoC=25)
   -> Seed 1/3 (seed=42)...
   -> Seed 2/3 (seed=2024)...
   -> Seed 3/3 (seed=12345)...
💡 h=3 | score=0.141959 | avg_iter=1491 | elapsed=41.9 min

 🚀 HORIZON 10
Data: train=1,337,236, test=362,057 | Features: 193 (DiffRoC=25)
   -> Seed 1/3 (seed=42)...
   -> Seed 2/3 (seed=2024)...
   -> Seed 3/3 (seed=12345)...
💡 h=10 | score=0.228232 | avg_iter=1122 | elapsed=32.3 min

 🚀 HORIZON 25
Data:

In [2]:
# Confirm submission saved
import os
print('Files saved:')
for f in ['submission.csv', 'submission_noclip.csv', 'oof_v38.csv',
          'diag1_scores.png', 'diag2_pred_vs_true.png',
          'diag3_temporal_residual.png', 'diag4_feature_importance.png']:
    size = os.path.getsize(f) // 1024 if os.path.exists(f) else 0
    status = f'{size} KB' if size else 'MISSING'
    print(f'  {f}: {status}')
print('\n✅ submission.csv sẵn sàng nộp!')

Files saved:
  submission.csv: 84442 KB
  submission_noclip.csv: 84442 KB
  oof_v38.csv: 13571 KB
  diag1_scores.png: 31 KB
  diag2_pred_vs_true.png: 89 KB
  diag3_temporal_residual.png: 199 KB
  diag4_feature_importance.png: 142 KB

✅ submission.csv sẵn sàng nộp!
